# Siamese Latent Graph Learning for Code Clones

A lightweight research model for XGLUE4. It pads AST node-type and
node-depth sequences, learns a small latent graph with attention, and uses the
latent Laplacian eigenvalues as a differentiable spectral representation.

The notebook trains on XGLUE4 and saves results, history, checkpoints, and
plots under `/kaggle/working/`.


## 0. Configuration

Attach the XGLUE4 clean export to this Kaggle notebook. The default is a full training run: every training pair is used. Set `MAX_TRAIN_PAIRS` to a smaller number only while debugging.


In [ ]:
from pathlib import Path

# Attach the XGLUE4 Kaggle dataset at the path configured below.
DATASET_KEYS = ("xglue4",)
KAGGLE_INPUT_BASE = Path("/kaggle/input/datasets/koushamoeini")
WORK_DIR = Path("/kaggle/working")

# Full training is the default: use every XGLUE4 train pair.
# Set a number only for a quick debugging run.
MAX_TRAIN_PAIRS = None
MAX_VALID_PAIRS = None
MAX_TEST_PAIRS = None

MAX_AST_NODES = 128
MAX_AST_DEPTH = 63
HIDDEN_DIM = 96
LATENT_NODES = 24
ATTENTION_HEADS = 4
ATTENTION_LAYERS = 1
DROPOUT = 0.10

OBJECTIVE = "hybrid"  # "bce", "contrastive", or "hybrid"
MARGIN = 0.25
HARD_NEGATIVE_WEIGHT = 2.0
EPOCHS = 12
BATCH_SIZE = 512
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 4
NUM_WORKERS = 0  # Kept at zero for stable Kaggle notebook workers.
USE_AMP = True
SEED = 42


## 1. Load Clean Exports and Build Fixed AST Inputs

The loader reads only `pairs.csv` and `graph_spectra.jsonl` from each portable export. It pads or truncates variable-size ASTs to `MAX_AST_NODES`; no Joern step is required in Kaggle.


In [ ]:
import copy
import csv
import gzip
import json
import random
import zipfile
from collections import Counter, deque
from contextlib import nullcontext
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print("Device:", DEVICE)


def is_gzip_file(path: Path) -> bool:
    with path.open("rb") as handle:
        return handle.read(2) == b"\x1f\x8b"


def open_text(path: Path):
    return gzip.open(path, "rt", encoding="utf-8") if is_gzip_file(path) else path.open("r", encoding="utf-8")


def dataset_root(dataset_key: str) -> Path:
    aliases = {"xglue4": ("xglue4",)}
    names = aliases.get(dataset_key.lower(), (dataset_key,))
    direct = [base / name for name in names for base in (KAGGLE_INPUT_BASE, Path("/kaggle/input"))]
    for candidate in direct:
        if candidate.exists():
            return candidate
    input_root = Path("/kaggle/input")
    matches = [path for path in input_root.rglob("*") if path.is_dir() and path.name.lower() in {name.lower() for name in names}]
    if matches:
        return sorted(matches, key=lambda path: (len(path.parts), str(path)))[0]
    raise FileNotFoundError(f"Could not locate the Kaggle input for {dataset_key}.")


def candidate_files(root: Path, names: tuple[str, ...]) -> list[Path]:
    matches = []
    for name in names:
        for candidate in (root / name,):
            if candidate.is_file():
                matches.append(candidate)
        matches.extend(path for path in root.rglob(name) if path.is_file())
        matches.extend(path for path in root.rglob(name + ".tmp") if path.is_file())
        matches.extend(path for path in root.rglob(name + ".gz.tmp") if path.is_file())
    return sorted(set(matches), key=lambda path: (len(path.relative_to(root).parts), len(path.name), str(path)))


def ensure_clean_root(dataset_key: str) -> Path:
    root = dataset_root(dataset_key)
    if candidate_files(root, ("pairs.csv.gz", "pairs.csv")):
        return root
    archives = sorted((path for path in root.rglob("*.zip") if path.is_file()), key=lambda path: (len(path.parts), path.name))
    if not archives:
        return root
    destination = WORK_DIR / f"{dataset_key}_clean_export"
    marker = destination / ".extracted"
    if not marker.exists():
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            archive.extractall(destination)
        marker.write_text(str(archives[0]), encoding="utf-8")
    return destination


def find_clean_file(root: Path, *names: str) -> Path:
    matches = candidate_files(root, tuple(names))
    if not matches:
        available = [str(path) for path in list(root.rglob("*"))[:30]]
        raise FileNotFoundError(f"Could not find {names} below {root}. First paths: {available}")
    return matches[0]


def load_pairs(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, compression="gzip" if is_gzip_file(path) else None, dtype={"left_id": str, "right_id": str, "split": str, "label": np.int64})
    required = {"split", "left_id", "right_id", "label"}
    if not required.issubset(frame.columns):
        raise ValueError(f"pairs file lacks columns {required - set(frame.columns)}")
    frame["label"] = frame["label"].astype(np.int64)
    return frame[["split", "left_id", "right_id", "label"]]


def limit_split(frame: pd.DataFrame, split: str, maximum: int | None, seed: int) -> pd.DataFrame:
    part = frame[frame.split == split].copy()
    if maximum is None or len(part) <= maximum:
        return part.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    chunks = []
    for label, group in part.groupby("label"):
        count = round(maximum * len(group) / len(part))
        chunks.append(group.sample(n=min(count, len(group)), random_state=seed + int(label)))
    sampled = pd.concat(chunks, ignore_index=True)
    if len(sampled) > maximum:
        sampled = sampled.sample(n=maximum, random_state=seed)
    return sampled.sample(frac=1.0, random_state=seed + 7).reset_index(drop=True)


def ast_depths(num_nodes: int, row: list, col: list) -> np.ndarray:
    children = [[] for _ in range(num_nodes)]
    indegree = np.zeros(num_nodes, dtype=np.int32)
    for parent, child in zip(row, col):
        try:
            parent, child = int(parent), int(child)
        except (TypeError, ValueError):
            continue
        if 0 <= parent < num_nodes and 0 <= child < num_nodes:
            children[parent].append(child)
            indegree[child] += 1
    depths = np.full(num_nodes, -1, dtype=np.int16)
    queue = deque(np.flatnonzero(indegree == 0).tolist() or [0])
    for root in queue:
        depths[root] = 0
    while queue:
        parent = queue.popleft()
        for child in children[parent]:
            if depths[child] < 0:
                depths[child] = min(depths[parent] + 1, MAX_AST_DEPTH)
                queue.append(child)
    depths[depths < 0] = 0
    return depths


@dataclass
class ASTStore:
    node_types: np.ndarray
    depths: np.ndarray
    masks: np.ndarray
    id_to_index: dict[str, int]
    vocabulary_size: int
    truncated_codes: int


def load_fixed_ast_store(graphs_path: Path, needed_ids: set[str]) -> ASTStore:
    ordered_ids = sorted(needed_ids)
    id_to_index = {code_id: index for index, code_id in enumerate(ordered_ids)}
    node_types = np.zeros((len(ordered_ids), MAX_AST_NODES), dtype=np.int32)
    depths = np.zeros((len(ordered_ids), MAX_AST_NODES), dtype=np.int16)
    masks = np.zeros((len(ordered_ids), MAX_AST_NODES), dtype=np.bool_)
    vocabulary = {"<pad>": 0, "<unk>": 1}
    seen = set()
    truncated = 0

    with open_text(graphs_path) as handle:
        for line in tqdm(handle, desc="Loading fixed AST inputs", unit="code"):
            if not line.strip():
                continue
            record = json.loads(line)
            code_id = str(record.get("code_id"))
            if code_id not in id_to_index:
                continue
            adjacency = record.get("graphs", {}).get("ast", {}).get("adjacency", {})
            raw_types = adjacency.get("node_types", [])
            num_nodes = min(int(adjacency.get("num_nodes", 0) or 0), len(raw_types))
            if num_nodes <= 0:
                continue
            full_depths = ast_depths(num_nodes, adjacency.get("row", []), adjacency.get("col", []))
            keep = min(num_nodes, MAX_AST_NODES)
            if num_nodes > MAX_AST_NODES:
                truncated += 1
            index = id_to_index[code_id]
            for position, node_type in enumerate(raw_types[:keep]):
                node_type = str(node_type)
                vocabulary.setdefault(node_type, len(vocabulary))
                node_types[index, position] = vocabulary.get(node_type, 1)
            depths[index, :keep] = full_depths[:keep]
            masks[index, :keep] = True
            seen.add(code_id)

    missing = needed_ids - seen
    if missing:
        raise RuntimeError(f"{len(missing):,} pair-referenced code ids have no AST record.")
    return ASTStore(node_types, depths, masks, id_to_index, len(vocabulary), truncated)


## 2. Shared Siamese Latent-Graph Model

Both code snippets pass through the same AST encoder. Attention pools AST nodes into a compact learned latent graph. The normalized latent Laplacian spectrum remains differentiable and becomes part of the learned code embedding.


In [ ]:
"""Lightweight Siamese latent-graph encoder for code-clone experiments.

The input is a fixed-size sequence of AST node types and depths. Cross
attention pools variable-size ASTs into a small learned latent graph. Its
normalized Laplacian eigenvalues are differentiable PyTorch features, so the
spectral representation is optimized jointly with clone supervision.
"""

import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class LatentGraphEncoder(nn.Module):
    def __init__(
        self,
        *,
        vocab_size: int,
        hidden_dim: int = 96,
        latent_nodes: int = 24,
        attention_heads: int = 4,
        attention_layers: int = 1,
        max_depth: int = 64,
        dropout: float = 0.10,
    ) -> None:
        super().__init__()
        if hidden_dim % attention_heads:
            raise ValueError("hidden_dim must be divisible by attention_heads.")

        self.latent_nodes = latent_nodes
        self.type_embedding = nn.Embedding(vocab_size, hidden_dim, padding_idx=0)
        self.depth_embedding = nn.Embedding(max_depth + 1, hidden_dim)
        self.input_norm = nn.LayerNorm(hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=attention_heads,
            dim_feedforward=hidden_dim * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.ast_attention = nn.TransformerEncoder(encoder_layer, num_layers=attention_layers)
        self.latent_queries = nn.Parameter(torch.empty(latent_nodes, hidden_dim))
        nn.init.normal_(self.latent_queries, std=hidden_dim**-0.5)
        self.pool_attention = nn.MultiheadAttention(hidden_dim, attention_heads, dropout=dropout, batch_first=True)
        self.latent_norm = nn.LayerNorm(hidden_dim)
        self.adjacency_query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.adjacency_key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.spectrum_projector = nn.Sequential(
            nn.LayerNorm(latent_nodes),
            nn.Linear(latent_nodes, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.output_norm = nn.LayerNorm(hidden_dim * 2)
        self.output_projector = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
        )

    def forward(
        self,
        node_types: torch.Tensor,
        depths: torch.Tensor,
        mask: torch.Tensor,
        *,
        return_spectrum: bool = False,
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        """Encode padded ASTs into L2-normalized latent spectral vectors."""
        if node_types.ndim != 2 or depths.shape != node_types.shape or mask.shape != node_types.shape:
            raise ValueError("node_types, depths, and mask must have matching [batch, nodes] shapes.")

        # Embedding indices must be int32/int64; the compact AST store uses int16 depths.
        node_types = node_types.long()
        depths = depths.long().clamp(min=0, max=self.depth_embedding.num_embeddings - 1)
        mask = mask.bool()
        tokens = self.input_norm(self.type_embedding(node_types) + self.depth_embedding(depths))
        ast_states = self.ast_attention(tokens, src_key_padding_mask=~mask)

        queries = self.latent_queries.unsqueeze(0).expand(node_types.size(0), -1, -1)
        latent, _ = self.pool_attention(queries, ast_states, ast_states, key_padding_mask=~mask, need_weights=False)
        latent = self.latent_norm(latent)

        query = self.adjacency_query(latent)
        key = self.adjacency_key(latent)
        scores = torch.matmul(query, key.transpose(1, 2)) / math.sqrt(query.size(-1))
        scores = 0.5 * (scores + scores.transpose(1, 2))
        adjacency = torch.sigmoid(scores)
        adjacency = adjacency * (1.0 - torch.eye(self.latent_nodes, device=adjacency.device, dtype=adjacency.dtype))

        degree = adjacency.sum(dim=-1).clamp_min(1e-6)
        normalized_adjacency = adjacency * degree.rsqrt().unsqueeze(-1) * degree.rsqrt().unsqueeze(-2)
        laplacian = torch.eye(self.latent_nodes, device=adjacency.device, dtype=adjacency.dtype).unsqueeze(0) - normalized_adjacency
        eigenvalues = torch.linalg.eigvalsh(laplacian)

        pooled_latent = latent.mean(dim=1)
        spectrum = self.spectrum_projector(eigenvalues)
        embedding = F.normalize(self.output_projector(self.output_norm(torch.cat([pooled_latent, spectrum], dim=1))), dim=1)
        return (embedding, eigenvalues) if return_spectrum else embedding


class SiameseLatentGraphModel(nn.Module):
    def __init__(self, encoder: LatentGraphEncoder) -> None:
        super().__init__()
        self.encoder = encoder
        self.logit_scale = nn.Parameter(torch.tensor(8.0))
        self.logit_bias = nn.Parameter(torch.tensor(0.0))

    def forward(
        self,
        left_types: torch.Tensor,
        left_depths: torch.Tensor,
        left_mask: torch.Tensor,
        right_types: torch.Tensor,
        right_depths: torch.Tensor,
        right_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        left = self.encoder(left_types, left_depths, left_mask)
        right = self.encoder(right_types, right_depths, right_mask)
        cosine = (left * right).sum(dim=1).clamp(-1.0, 1.0)
        logits = self.logit_scale.clamp(1.0, 30.0) * cosine + self.logit_bias
        return logits, cosine


def clone_loss(
    logits: torch.Tensor,
    cosine: torch.Tensor,
    labels: torch.Tensor,
    *,
    objective: str = "hybrid",
    margin: float = 0.25,
    hard_negative_weight: float = 2.0,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    """Combine calibrated pair BCE with a contrastive metric objective."""
    labels = labels.float()
    bce = F.binary_cross_entropy_with_logits(logits, labels)
    positive = labels * (1.0 - cosine).square()
    negative_hinge = (1.0 - labels) * F.relu(cosine - margin).square()
    hard_weight = 1.0 + (hard_negative_weight - 1.0) * (1.0 - labels) * (cosine.detach() > margin).float()
    metric = (positive + hard_weight * negative_hinge).mean()

    if objective == "bce":
        loss = bce
    elif objective == "contrastive":
        loss = metric
    elif objective == "hybrid":
        loss = bce + metric
    else:
        raise ValueError("objective must be one of: bce, contrastive, hybrid.")
    return loss, {"bce": bce.detach(), "metric": metric.detach()}


## 3. Pair Objective, Hard Negatives, and Training Loop

The hybrid objective combines clone classification with a cosine-margin contrastive term. Non-clone pairs that remain too similar within a batch receive additional weight, making them in-batch hard negatives.


In [ ]:
class ASTPairDataset(Dataset):
    def __init__(self, pairs: pd.DataFrame, store: ASTStore) -> None:
        left = pairs.left_id.map(store.id_to_index)
        right = pairs.right_id.map(store.id_to_index)
        if left.isna().any() or right.isna().any():
            raise RuntimeError("A selected pair references an AST outside the store.")
        self.left = left.to_numpy(np.int64)
        self.right = right.to_numpy(np.int64)
        self.labels = pairs.label.to_numpy(np.float32)
        self.store = store

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int):
        left, right = self.left[index], self.right[index]
        return (
            torch.from_numpy(self.store.node_types[left]),
            torch.from_numpy(self.store.depths[left]),
            torch.from_numpy(self.store.masks[left]),
            torch.from_numpy(self.store.node_types[right]),
            torch.from_numpy(self.store.depths[right]),
            torch.from_numpy(self.store.masks[right]),
            torch.tensor(self.labels[index], dtype=torch.float32),
        )


def make_loader(pairs: pd.DataFrame, store: ASTStore, shuffle: bool) -> DataLoader:
    return DataLoader(
        ASTPairDataset(pairs, store),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
    )


def autocast_context():
    return torch.amp.autocast("cuda", enabled=(DEVICE == "cuda" and USE_AMP)) if DEVICE == "cuda" else nullcontext()


def binary_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    prediction = (scores >= threshold).astype(np.int64)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, prediction, average="binary", zero_division=0)
    return {"P": float(precision), "R": float(recall), "F1": float(f1), "Acc": float(accuracy_score(labels, prediction))}


def best_threshold(labels: np.ndarray, scores: np.ndarray) -> tuple[float, dict]:
    candidates = np.unique(np.concatenate([np.quantile(scores, np.linspace(0.0, 1.0, 201)), np.asarray([0.5])]))
    best = (0.5, binary_metrics(labels, scores, 0.5))
    for threshold in candidates:
        metrics = binary_metrics(labels, scores, float(threshold))
        if (metrics["F1"], metrics["Acc"]) > (best[1]["F1"], best[1]["Acc"]):
            best = (float(threshold), metrics)
    return best


@torch.no_grad()
def predict(model: SiameseLatentGraphModel, loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    scores, labels = [], []
    for batch in loader:
        batch = [value.to(DEVICE, non_blocking=True) for value in batch]
        with autocast_context():
            logits, _ = model(*batch[:-1])
        scores.append(torch.sigmoid(logits).float().cpu().numpy())
        labels.append(batch[-1].long().cpu().numpy())
    return np.concatenate(labels), np.concatenate(scores)


def train_model(model: SiameseLatentGraphModel, train_loader: DataLoader, valid_loader: DataLoader) -> tuple[SiameseLatentGraphModel, pd.DataFrame, float, dict]:
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda" and USE_AMP)) if DEVICE == "cuda" else None
    history = []
    best_state, best_f1, stale = None, -1.0, 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        progress = tqdm(train_loader, desc=f"epoch {epoch:02d}", leave=False)
        for batch in progress:
            batch = [value.to(DEVICE, non_blocking=True) for value in batch]
            optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                logits, cosine = model(*batch[:-1])
                loss, parts = clone_loss(logits, cosine, batch[-1], objective=OBJECTIVE, margin=MARGIN, hard_negative_weight=HARD_NEGATIVE_WEIGHT)
            if scaler is None:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            else:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            losses.append(float(loss.detach().cpu()))
            progress.set_postfix(loss=f"{losses[-1]:.4f}")

        valid_labels, valid_scores = predict(model, valid_loader)
        threshold, valid_metrics = best_threshold(valid_labels, valid_scores)
        row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "threshold": threshold, **{f"valid_{key}": value for key, value in valid_metrics.items()}}
        history.append(row)
        print({key: round(value, 5) if isinstance(value, float) else value for key, value in row.items()})
        if valid_metrics["F1"] > best_f1:
            best_f1, stale = valid_metrics["F1"], 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
            if stale >= PATIENCE:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), history[np.argmax(pd.DataFrame(history)["valid_F1"].to_numpy())]["threshold"], {"BestValidF1": best_f1}


def run_dataset(dataset_key: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    root = ensure_clean_root(dataset_key)
    pairs_path = find_clean_file(root, "pairs.csv.gz", "pairs.csv")
    graphs_path = find_clean_file(root, "graph_spectra.jsonl.gz", "graph_spectra.jsonl")
    pairs = load_pairs(pairs_path)
    train_pairs = limit_split(pairs, "train", MAX_TRAIN_PAIRS, SEED)
    valid_pairs = limit_split(pairs, "valid", MAX_VALID_PAIRS, SEED + 1)
    test_pairs = limit_split(pairs, "test", MAX_TEST_PAIRS, SEED + 2)
    needed_ids = set(train_pairs.left_id) | set(train_pairs.right_id) | set(valid_pairs.left_id) | set(valid_pairs.right_id) | set(test_pairs.left_id) | set(test_pairs.right_id)
    print(f"\n{dataset_key.upper()}: pairs train/valid/test = {len(train_pairs):,}/{len(valid_pairs):,}/{len(test_pairs):,}; code ids = {len(needed_ids):,}")
    store = load_fixed_ast_store(graphs_path, needed_ids)
    print(f"AST vocab={store.vocabulary_size:,}; fixed nodes={MAX_AST_NODES}; truncated ASTs={store.truncated_codes:,}")

    train_loader = make_loader(train_pairs, store, shuffle=True)
    valid_loader = make_loader(valid_pairs, store, shuffle=False)
    test_loader = make_loader(test_pairs, store, shuffle=False)
    encoder = LatentGraphEncoder(vocab_size=store.vocabulary_size, hidden_dim=HIDDEN_DIM, latent_nodes=LATENT_NODES, attention_heads=ATTENTION_HEADS, attention_layers=ATTENTION_LAYERS, max_depth=MAX_AST_DEPTH, dropout=DROPOUT)
    model = SiameseLatentGraphModel(encoder)
    parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    print("Trainable parameters:", f"{parameters:,}")
    model, history, threshold, best = train_model(model, train_loader, valid_loader)
    test_labels, test_scores = predict(model, test_loader)
    test_metrics = binary_metrics(test_labels, test_scores, threshold)
    result = pd.DataFrame([{
        "Dataset": dataset_key.upper(), "Method": "Siamese Latent Graph (AST)", "Objective": OBJECTIVE,
        "P": test_metrics["P"], "R": test_metrics["R"], "F1": test_metrics["F1"], "Acc": test_metrics["Acc"],
        "BestValidF1": best["BestValidF1"], "Threshold": threshold, "TrainableParameters": parameters,
        "TrainPairs": len(train_pairs), "ValidPairs": len(valid_pairs), "TestPairs": len(test_pairs),
        "ASTVocabulary": store.vocabulary_size, "TruncatedASTs": store.truncated_codes,
    }])
    history.insert(0, "Dataset", dataset_key.upper())
    result_path = WORK_DIR / f"{dataset_key}_latent_graph_results.csv"
    history_path = WORK_DIR / f"{dataset_key}_latent_graph_history.csv"
    checkpoint_path = WORK_DIR / f"{dataset_key}_latent_graph_model.pt"
    result.to_csv(result_path, index=False)
    history.to_csv(history_path, index=False)
    torch.save({"state_dict": model.state_dict(), "config": {"max_ast_nodes": MAX_AST_NODES, "hidden_dim": HIDDEN_DIM, "latent_nodes": LATENT_NODES, "attention_layers": ATTENTION_LAYERS, "objective": OBJECTIVE}, "threshold": threshold, "vocabulary_size": store.vocabulary_size}, checkpoint_path)
    print("Saved:", result_path, history_path, checkpoint_path)
    del model, train_loader, valid_loader, test_loader, store
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return result, history


## 4. Full Training and Evaluation for XGLUE4

This cell trains on XGLUE4, selects a decision threshold only from validation data, evaluates once on the test split, and saves CSVs and a checkpoint under `/kaggle/working/`.


In [ ]:
all_results, all_history = [], []
for dataset_key in DATASET_KEYS:
    result, history = run_dataset(dataset_key)
    all_results.append(result)
    all_history.append(history)

results_df = pd.concat(all_results, ignore_index=True)
history_df = pd.concat(all_history, ignore_index=True)
display(results_df.style.format({"P": "{:.4f}", "R": "{:.4f}", "F1": "{:.4f}", "Acc": "{:.4f}", "BestValidF1": "{:.4f}", "Threshold": "{:.4f}"}))
results_df.to_csv(WORK_DIR / "xglue4_latent_graph_results.csv", index=False)
history_df.to_csv(WORK_DIR / "xglue4_latent_graph_history.csv", index=False)
print("XGLUE4 CSVs saved under", WORK_DIR)


## 5. Learning Curves


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for dataset, part in history_df.groupby("Dataset"):
    axes[0].plot(part["epoch"], part["train_loss"], marker="o", label=dataset)
    axes[1].plot(part["epoch"], part["valid_F1"], marker="o", label=dataset)
axes[0].set(title="Latent graph training loss", xlabel="Epoch", ylabel="Loss")
axes[1].set(title="Latent graph validation F1", xlabel="Epoch", ylabel="F1")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
fig.tight_layout()
plot_path = WORK_DIR / "latent_graph_training_curves.png"
fig.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", plot_path)


## Suggested Ablations

Keep the data splits fixed. Compare `OBJECTIVE` values (`"bce"`, `"contrastive"`, `"hybrid"`), then vary `LATENT_NODES` (16, 24, 32), `HIDDEN_DIM` (64, 96, 128), and `ATTENTION_LAYERS` (1, 2). Record F1, precision, recall, parameter count, and runtime for each setting.
